https://python.langchain.com/docs/use_cases/sql/quickstart/

### **Test the sqldb**

In [27]:
from langchain_community.utilities import SQLDatabase
from pyprojroot import here
import warnings
warnings.filterwarnings("ignore")

**Connecting to the sqldb**

In [125]:
db_path = str(here("data")) + "/csv_xlsx_sqldb.db"
db = SQLDatabase.from_uri(f"sqlite:///{db_path}")

In [126]:
db

In [127]:
# validate the connection to the vectordb
print(db.dialect)

sqlite


In [128]:
print(db.get_usable_table_names())

['The_Cancer_data_1500_V2', 'diabetes', 'heart']


In [129]:

db._execute("SELECT * FROM The_Cancer_data_1500_V2 LIMIT 10;")

[{'Age': 58,
  'Gender': 1,
  'BMI': 16.085313321370478,
  'Smoking': 0,
  'GeneticRisk': 1,
  'PhysicalActivity': 8.146250560259173,
  'AlcoholIntake': 4.148219026764642,
  'CancerHistory': 1,
  'Diagnosis': 1},
 {'Age': 71,
  'Gender': 0,
  'BMI': 30.82878438985056,
  'Smoking': 0,
  'GeneticRisk': 1,
  'PhysicalActivity': 9.361630415509964,
  'AlcoholIntake': 3.519683335172577,
  'CancerHistory': 0,
  'Diagnosis': 0},
 {'Age': 48,
  'Gender': 1,
  'BMI': 38.78508355516642,
  'Smoking': 0,
  'GeneticRisk': 2,
  'PhysicalActivity': 5.1351786674177005,
  'AlcoholIntake': 4.728367685254023,
  'CancerHistory': 0,
  'Diagnosis': 1},
 {'Age': 34,
  'Gender': 0,
  'BMI': 30.04029550365828,
  'Smoking': 0,
  'GeneticRisk': 0,
  'PhysicalActivity': 9.50279223611407,
  'AlcoholIntake': 2.04463617878336,
  'CancerHistory': 0,
  'Diagnosis': 0},
 {'Age': 62,
  'Gender': 1,
  'BMI': 35.47972148566976,
  'Smoking': 0,
  'GeneticRisk': 0,
  'PhysicalActivity': 5.356889704560298,
  'AlcoholIntake': 

### **Test the access to the environment variables**

In [34]:
import os
from langchain_openai import ChatOpenAI


os.environ['GITHUB_TOKEN'] = "GITHUB_TOKEN_REMOVED"  # Replace with your actual GitHub token
token = os.environ.get("GITHUB_TOKEN")
endpoint = "https://models.github.ai/inference"
model_name = "openai/gpt-4.1-mini" 

if not token:
    raise ValueError("GITHUB_TOKEN environment variable not set. Please provide a valid token.")

llm =ChatOpenAI(
    model_name=model_name,
    openai_api_key=token,
    openai_api_base=endpoint,
    temperature=0.5,
)

In [35]:
import os
print(os.getenv("GITHUB_TOKEN"))


GITHUB_TOKEN_REMOVED


### **Test your GPT model**

In [36]:
from langchain_core.messages import HumanMessage, AIMessage, SystemMessage

response = llm.invoke([
    HumanMessage(content="What's the capital of Bangladesh?")
])

print(response.content)

The capital of Bangladesh is Dhaka.


In [13]:
response

AIMessage(content='The capital of Bangladesh is Dhaka.', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 9, 'prompt_tokens': 13, 'total_tokens': 22, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-4.1-mini-2025-04-14', 'system_fingerprint': 'fp_3dcd5944f5', 'id': 'chatcmpl-CYczNnu0RxYeKp7gGJUAQrfN7H6nv', 'finish_reason': 'stop', 'logprobs': None}, id='lc_run--a8602390-380b-4131-bd61-244b60659216-0', usage_metadata={'input_tokens': 13, 'output_tokens': 9, 'total_tokens': 22, 'input_token_details': {'audio': 0, 'cache_read': 0}, 'output_token_details': {'audio': 0, 'reasoning': 0}})

### **1. SQL query chain**

In [39]:
import os
import warnings
from pyprojroot import here
from langchain_openai import ChatOpenAI
from langchain_community.utilities import SQLDatabase
from langchain_community.agent_toolkits import SQLDatabaseToolkit
from langchain_community.agent_toolkits.sql.base import create_sql_agent

In [63]:
# ------------------ AGENT SETUP ------------------
toolkit = SQLDatabaseToolkit(db=db, llm=llm)
agent = create_sql_agent(llm=llm, toolkit=toolkit, verbose=True)

# ------------------ QUERY ------------------
response = agent.invoke({"input": "How many employees are there?"})
print(response["output"])



> Entering new SQL Agent Executor chain...
Action: sql_db_list_tables
Action Input: 
Album, Artist, Customer, Employee, Genre, Invoice, InvoiceLine, MediaType, Playlist, PlaylistTrack, TrackThought: There is a table named "Employee" which likely contains the employee data. I should check the schema of the Employee table to confirm the relevant column(s) for counting employees.
Action: sql_db_schema
Action Input: Employee
CREATE TABLE "Employee" (
	"EmployeeId" INTEGER NOT NULL, 
	"LastName" NVARCHAR(20) NOT NULL, 
	"FirstName" NVARCHAR(20) NOT NULL, 
	"Title" NVARCHAR(30), 
	"ReportsTo" INTEGER, 
	"BirthDate" DATETIME, 
	"HireDate" DATETIME, 
	"Address" NVARCHAR(70), 
	"City" NVARCHAR(40), 
	"State" NVARCHAR(40), 
	"Country" NVARCHAR(40), 
	"PostalCode" NVARCHAR(10), 
	"Phone" NVARCHAR(24), 
	"Fax" NVARCHAR(24), 
	"Email" NVARCHAR(60), 
	PRIMARY KEY ("EmployeeId"), 
	FOREIGN KEY("ReportsTo") REFERENCES "Employee" ("EmployeeId")
)

/*
3 rows from Employee table:
EmployeeId	LastName	Fi

In [83]:
import re
from io import StringIO

# ------------------ CALLBACK HANDLER ------------------
class SQLCaptureHandler:
    """Captures verbose output of the agent (works in LangChain 1.0.2)."""
    def __init__(self):
        self.buffer = StringIO()

    def on_llm_new_token(self, token, **kwargs):
        self.buffer.write(token)

    def get_text(self):
        return self.buffer.getvalue()

# ------------------ SQL EXTRACTOR ------------------
def extract_sql_query(agent_output: str) -> str | None:
    """Extract the last SQL query from agent verbose text."""
    if not isinstance(agent_output, str):
        return None
    pattern = re.compile(r"Action Input:\s*(.*?;)", re.IGNORECASE | re.DOTALL)
    matches = pattern.findall(agent_output)
    return matches[-1].strip() if matches else None

# ------------------ AGENT RUNNER ------------------
def run_agent_with_sql_capture(agent, user_question: str):
    """Run SQL agent and return (SQL query, final answer)."""
    capture = SQLCaptureHandler()
    # Pass our simple handler in the callbacks list
    response = agent.invoke({"input": user_question}, config={"callbacks": [capture]})
    
    verbose_text = capture.get_text()
    sql_query = extract_sql_query(verbose_text)
    final_answer = response.get("output", "")
    
    return sql_query, final_answer


In [85]:
import sys
from io import StringIO

# Redirect stdout
old_stdout = sys.stdout
sys.stdout = mystdout = StringIO()

# Run your agent in verbose mode
agent.verbose = True
response = agent.invoke({"input": "How many employees are there?"})

# Reset stdout
sys.stdout = old_stdout

# Get the full verbose text
verbose_text = mystdout.getvalue()

# Extract SQL
sql_query = extract_sql_query(verbose_text)
final_answer = response.get("output", "")

print("🧾 Extracted SQL Query:\n", sql_query)
print("\n✅ Final Answer:\n", final_answer)


🧾 Extracted SQL Query:
 SELECT COUNT(*) AS employee_count FROM Employee;

✅ Final Answer:
 There are 8 employees.


Execute the query to make sure it’s valid

In [87]:
db.run(sql_query)

'[(8,)]'

In [88]:
db._execute(sql_query)

[{'employee_count': 8}]

<!-- 
SQL agent is built as:

<!-- RunnableSequence([
    RunnableAssign(...),
    PromptTemplate(...),
    LLM(...),
    OutputParser(...)
]) --> -->


In [120]:
from langchain.chains import create_sql_query_chain
from langchain_community.tools.sql_database.tool import QuerySQLDataBaseTool
from langchain.chains.sql_database.base import extract_sql_query_simple

# create LLM → SQL query chain
write_query = create_sql_query_chain(llm, db)

# create execution tool
execute_query = QuerySQLDataBaseTool(db=db)

# combine the components
chain = write_query | extract_sql_query_simple | execute_query

result = chain.invoke({"question": "How many employees are there?"})
print(result)


ModuleNotFoundError: No module named 'langchain.chains'

In [119]:
prompt_template = agent_executor.agent.runnable.middle[0].template
print(prompt_template)


AttributeError: 'ChatPromptTemplate' object has no attribute 'template'

### **Add QuerySQLDataBaseTool to the chain**
Execute SQL query

**This is the most dangerous part of creating a SQL chain.** Consider carefully if it is OK to run automated queries over your data. Minimize the database connection permissions as much as possible. Consider adding a human approval step to you chains before query execution (see below).

We can use the QuerySQLDatabaseTool to easily add query execution to our chain:

In [ ]:
# userquestion->create_sql_query_chain->response->Regex(extract_sql_query_simple)->query->db.run(query)->db._execute(query)

In [113]:
from langchain.chains.sql_database.query import create_sql_query_chain
from langchain_community.tools.sql_database.tool import QuerySQLDataBaseTool
from langchain.chains.sql_database.base import extract_sql_query_simple

# Step 1. Create the query-writing chain (LLM → SQL)
write_query = create_sql_query_chain(llm, db)

# Step 2. Create the tool that actually executes SQL on your database
execute_query = QuerySQLDataBaseTool(db=db)

# Step 3. Connect them into one functional pipeline
chain = write_query | extract_sql_query_simple | execute_query

# Step 4. Run a test question
result = chain.invoke({"question": "How many employees are there?"})
print(result)


ModuleNotFoundError: No module named 'langchain.chains'

1.0.1
0.4


In [114]:
chain.invoke({"question": "Give me the names of all employees in the company (dont use any limit)"})

AttributeError: 'SQLDatabaseToolkit' object has no attribute 'invoke'

In [ ]:
chain.invoke({"question": "GIve me all employee title ? (dont use any limit)"})

"[('General Manager',), ('Sales Manager',), ('Sales Support Agent',), ('IT Manager',), ('IT Staff',)]"

### **Answer the question in a user friendly manner**

In [ ]:
chain=write_query |extract_sql_query_simple

In [ ]:
from operator import itemgetter
from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import PromptTemplate
from langchain_core.runnables import RunnablePassthrough

answer_prompt = PromptTemplate.from_template(
    """Given the following user question, corresponding SQL query, and SQL result, answer the user question.

Question: {question}
SQL Query: {query}
SQL Result: {result}

Answer: """
)

answer = answer_prompt | llm | StrOutputParser()

query=chain.invoke({"question": "How many employees are there"})
sql_query_response = db._execute(query)

answer.invoke({"question": "How many employees are there","query": query, "result": sql_query_response})

'There are 8 employees.'

In [ ]:
query=chain.invoke({"question": "Give me the names of all employees in the company (dont use any limit)"})
sql_query_response = db._execute(query)

answer.invoke({"question": "Give me the names of all employees in the company (dont use any limit)","query": query, "result": sql_query_response})

'The names of all employees in the company are:\n\n- Andrew Adams  \n- Nancy Edwards  \n- Jane Peacock  \n- Margaret Park  \n- Steve Johnson  \n- Michael Mitchell  \n- Robert King  \n- Laura Callahan'

### **2. Agents**

Agent which provides a more flexible way of interacting with SQL databases. The main advantages of using the SQL Agent are:

- It can answer questions based on the databases’ schema as well as on the databases’ content (like describing a specific table).
- It can recover from errors by running a generated query, catching the traceback and regenerating it correctly.
- It can answer questions that require multiple dependent queries.
- It will save tokens by only considering the schema from relevant tables.

To initialize the agent, we use create_sql_agent function. This agent contains the SQLDatabaseToolkit which contains tools to:

- Create and execute queries
- Check query syntax
- Retrieve table descriptions
- …

In [115]:
from langchain_community.agent_toolkits import create_sql_agent

agent_executor = create_sql_agent(llm, db=db, agent_type="openai-tools", verbose=True)

In [116]:
agent_executor.invoke(
    {
        "input": "List the total sales per country. Which country's customers spent the most?"
    }
)



> Entering new SQL Agent Executor chain...

Invoking: `sql_db_list_tables` with `{}`


Album, Artist, Customer, Employee, Genre, Invoice, InvoiceLine, MediaType, Playlist, PlaylistTrack, Track
Invoking: `sql_db_schema` with `{'table_names': 'Customer, Invoice'}`



CREATE TABLE "Customer" (
	"CustomerId" INTEGER NOT NULL, 
	"FirstName" NVARCHAR(40) NOT NULL, 
	"LastName" NVARCHAR(20) NOT NULL, 
	"Company" NVARCHAR(80), 
	"Address" NVARCHAR(70), 
	"City" NVARCHAR(40), 
	"State" NVARCHAR(40), 
	"Country" NVARCHAR(40), 
	"PostalCode" NVARCHAR(10), 
	"Phone" NVARCHAR(24), 
	"Fax" NVARCHAR(24), 
	"Email" NVARCHAR(60) NOT NULL, 
	"SupportRepId" INTEGER, 
	PRIMARY KEY ("CustomerId"), 
	FOREIGN KEY("SupportRepId") REFERENCES "Employee" ("EmployeeId")
)

/*
3 rows from Customer table:
CustomerId	FirstName	LastName	Company	Address	City	State	Country	PostalCode	Phone	Fax	Email	SupportRepId
1	Luís	Gonçalves	Embraer - Empresa Brasileira de Aeronáutica S.A.	Av. Brigadeiro Faria Lima, 2170	São José

{'input': "List the total sales per country. Which country's customers spent the most?",
 'output': 'The total sales per country are as follows (top 10):\n- USA: 523.06\n- Canada: 303.96\n- France: 195.10\n- Brazil: 190.10\n- Germany: 156.48\n- United Kingdom: 112.86\n- Czech Republic: 90.24\n- Portugal: 77.24\n- India: 75.26\n- Chile: 46.62\n\nThe customers from the USA spent the most.'}

In [117]:
agent_executor.invoke({"input": "Describe the playlisttrack table"})
# agent_executor.invoke("Describe the playlisttrack table")



> Entering new SQL Agent Executor chain...

Invoking: `sql_db_list_tables` with `{}`


Album, Artist, Customer, Employee, Genre, Invoice, InvoiceLine, MediaType, Playlist, PlaylistTrack, Track
Invoking: `sql_db_schema` with `{'table_names': 'PlaylistTrack'}`



CREATE TABLE "PlaylistTrack" (
	"PlaylistId" INTEGER NOT NULL, 
	"TrackId" INTEGER NOT NULL, 
	PRIMARY KEY ("PlaylistId", "TrackId"), 
	FOREIGN KEY("TrackId") REFERENCES "Track" ("TrackId"), 
	FOREIGN KEY("PlaylistId") REFERENCES "Playlist" ("PlaylistId")
)

/*
3 rows from PlaylistTrack table:
PlaylistId	TrackId
1	3402
1	3389
1	3390
*/The PlaylistTrack table has two columns: PlaylistId and TrackId. Both columns are integers and together they form the primary key of the table. PlaylistId is a foreign key that references the Playlist table, and TrackId is a foreign key that references the Track table. The table represents the many-to-many relationship between playlists and tracks, indicating which tracks belong to which playlist

{'input': 'Describe the playlisttrack table',
 'output': 'The PlaylistTrack table has two columns: PlaylistId and TrackId. Both columns are integers and together they form the primary key of the table. PlaylistId is a foreign key that references the Playlist table, and TrackId is a foreign key that references the Track table. The table represents the many-to-many relationship between playlists and tracks, indicating which tracks belong to which playlists.'}

In [118]:
agent_executor.invoke({"input": "Tell me about the Artist table. What is the primary key?"})



> Entering new SQL Agent Executor chain...

Invoking: `sql_db_list_tables` with `{}`


Album, Artist, Customer, Employee, Genre, Invoice, InvoiceLine, MediaType, Playlist, PlaylistTrack, Track
Invoking: `sql_db_schema` with `{'table_names': 'Artist'}`



CREATE TABLE "Artist" (
	"ArtistId" INTEGER NOT NULL, 
	"Name" NVARCHAR(120), 
	PRIMARY KEY ("ArtistId")
)

/*
3 rows from Artist table:
ArtistId	Name
1	AC/DC
2	Accept
3	Aerosmith
*/The Artist table has two columns: ArtistId and Name. The primary key of the Artist table is ArtistId.

> Finished chain.


{'input': 'Tell me about the Artist table. What is the primary key?',
 'output': 'The Artist table has two columns: ArtistId and Name. The primary key of the Artist table is ArtistId.'}